In [ ]:
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

In [ ]:
df = pd.read_parquet("../data/raw/all_laps.parquet")

In [ ]:
for col in ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']:
    df[col] = df[col].dt.total_seconds()

df['IsPersonalBest'] = df['IsPersonalBest'].fillna(False).astype(bool)
df['DriverNumber'] = df['DriverNumber'].astype(int)

In [ ]:
driver_names = {
    'VER': 'Verstappen', 'PER': 'Perez',  'SAI': 'Sainz',
    'LEC': 'Leclerc',   'RUS': 'Russell', 'NOR': 'Norris',
    'HAM': 'Hamilton',  'PIA': 'Piastri',  'ALO': 'Alonso',
    'STR': 'Stroll',    'ZHO': 'Zhou',     'MAG': 'Magnussen',
    'RIC': 'Ricciardo', 'TSU': 'Tsunoda',  'ALB': 'Albon',
    'HUL': 'Hulkenberg','OCO': 'Ocon',     'GAS': 'Gasly',
    'BOT': 'Bottas',    'SAR': 'Sargeant', 'BEA': 'Bearman',
    'COL': 'Colapinto', 'LAW': 'Lawson',   'DOO': 'Doohan',
    'MSC': 'Schumacher','LAT': 'Latifi',   'VET': 'Vettel',
    'DEV': 'Devereux',
}
df['Driver'] = df['Driver'].map(driver_names)

In [ ]:
df['PitIn']  = df['PitInTime'].notna().astype(int)
df['PitOut'] = df['PitOutTime'].notna().astype(int)

In [ ]:
df = df[df['Deleted'] == False]
df = df[df['IsAccurate'] == True]
df = df[df['Driver'].notna()]
df = df[df['LapTime'].notna()]
df = df[df['LapTime'] < 150]

In [ ]:
df['TyreLife'] = df.groupby(['Year', 'RoundNumber', 'Driver'])['TyreLife'].ffill()
df['Position'] = df.groupby(['Year', 'RoundNumber', 'Driver'])['Position'].ffill()

for col in ['SpeedI1', 'SpeedI2', 'SpeedST']:
    df[col] = df.groupby(['Year', 'RoundNumber', 'Driver'])[col].transform(
        lambda x: x.fillna(x.mean())
    )

In [ ]:
df = df.drop(columns=[
    'PitInTime', 'PitOutTime',
    'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
    'SpeedFL', 'LapStartDate', 'DeletedReason',
])
df = df.dropna(subset=['Sector1Time', 'Sector2Time', 'Sector3Time'])

In [ ]:
df['Compound'] = df['Compound'].replace('WET', 'INTERMEDIATE')
df = df[df['Compound'] != 'None']

In [ ]:
final_laps = (
    df.sort_values('LapNumber')
    .groupby(['Year', 'RoundNumber', 'Driver'])
    .last()
    .reset_index()
)

p1_drivers = final_laps[final_laps['Position'] == 1].copy()
winners = (
    p1_drivers
    .sort_values('LapNumber', ascending=False)
    .groupby(['Year', 'RoundNumber'])
    .first()
    .reset_index()[['Year', 'RoundNumber', 'Driver']]
)
winners['Won'] = 1

In [ ]:
np.random.seed(45)

df_train = df[df['Year'].isin([2022, 2023])].copy()
df_val   = df[(df['Year'] == 2024) & (df['RoundNumber'] <= 12)].copy()
df_test  = df[(df['Year'] == 2024) & (df['RoundNumber'] > 12)].copy()

In [ ]:
df_train = df_train.merge(winners, on=['Year', 'RoundNumber', 'Driver'], how='left')
df_val   = df_val.merge(winners,   on=['Year', 'RoundNumber', 'Driver'], how='left')
df_test  = df_test.merge(winners,  on=['Year', 'RoundNumber', 'Driver'], how='left')

df_train['Won'] = df_train['Won'].fillna(0).astype(int)
df_val['Won']   = df_val['Won'].fillna(0).astype(int)
df_test['Won']  = df_test['Won'].fillna(0).astype(int)

y_train = df_train.pop('Won').values
y_val   = df_val.pop('Won').values
y_test  = df_test.pop('Won').values

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Race completion %
    total_laps = df.groupby(['Year', 'RoundNumber'])['LapNumber'].transform('max')
    df['RacePct'] = df['LapNumber'] / total_laps

    # Compound ordinal (grip level: SOFT=0 ... INTERMEDIATE=3)
    compound_order = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2, 'INTERMEDIATE': 3}
    df['CompoundOrdinal'] = df['Compound'].map(compound_order)

    # Average trap speed
    df['SpeedAvg'] = (df['SpeedI1'] + df['SpeedI2'] + df['SpeedST']) / 3

    # Lap time delta from driver's race median pace
    df['LapTimeDelta'] = df.groupby(['Year', 'RoundNumber', 'Driver'])['LapTime'].transform(
        lambda x: x - x.median()
    )

    # Positions gained/lost from grid (proxy: first observed lap)
    start_pos = (
        df.sort_values('LapNumber')
        .groupby(['Year', 'RoundNumber', 'Driver'])['Position']
        .first()
        .rename('StartPosition')
        .reset_index()
    )
    df = df.merge(start_pos, on=['Year', 'RoundNumber', 'Driver'], how='left')
    df['PositionGain'] = df['StartPosition'] - df['Position']

    return df

df_train = engineer_features(df_train)
df_val   = engineer_features(df_val)
df_test  = engineer_features(df_test)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df['LapTime'][df['LapTime'] < 150].hist(bins=50, ax=ax, color='#1e3a5f', edgecolor='white')
ax.set_title('Lap Time Distribution (2022–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Lap Time (seconds)')
ax.set_ylabel('Count')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df['Compound'].value_counts().plot(kind='bar', ax=ax, color='#e8002d', edgecolor='white')
ax.set_title('Tyre Compound Usage (2022–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Compound')
ax.set_ylabel('Lap Count')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df['Position'].value_counts().sort_index().plot(kind='bar', ax=ax1,
    color='#f39c12', edgecolor='white')
ax1.set_title('Position Distribution (2022–2024)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Race Position')
ax1.set_ylabel('Lap Count')
ax1.tick_params(axis='x', rotation=0)
ax1.grid(axis='y', alpha=0.3)

df['TyreLife'].hist(bins=40, ax=ax2, color='#2ecc71', edgecolor='white')
ax2.set_title('Tyre Age Distribution (2022–2024)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Laps on Tyre')
ax2.set_ylabel('Count')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
lap2 = (
    df[df['LapNumber'] == 2.0][['Year', 'RoundNumber', 'Driver', 'Position']]
    .copy()
    .rename(columns={'Position': 'StartPosition'})
    .dropna(subset=['StartPosition'])
)
lap2['StartPosition'] = lap2['StartPosition'].astype(int)

merged = lap2.merge(winners, on=['Year', 'RoundNumber', 'Driver'], how='left')
merged['Won'] = merged['Won'].fillna(0)
win_rate = merged.groupby('StartPosition')['Won'].mean().reset_index()
win_rate.columns = ['StartPosition', 'WinRate']

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(
    win_rate['StartPosition'], win_rate['WinRate'] * 100,
    color=['#e8002d' if p == 1 else '#1e3a5f' for p in win_rate['StartPosition']],
    edgecolor='white', linewidth=0.5,
)
ax.set_xlabel('Position at Lap 2', fontsize=12)
ax.set_ylabel('Win Rate %', fontsize=12)
ax.set_title('Win Rate by Position at Lap 2 (2022–2024)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xticks(win_rate['StartPosition'])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
bahrain = df[(df['Year'] == 2023) & (df['RoundNumber'] == 1)].copy()

fig, ax = plt.subplots(figsize=(14, 7))
for driver in bahrain['Driver'].unique():
    d = bahrain[bahrain['Driver'] == driver].sort_values('LapNumber')
    ax.plot(d['LapNumber'], d['Position'], linewidth=1.5, label=driver, alpha=0.8)

ax.set_xlabel('Lap Number', fontsize=12)
ax.set_ylabel('Position', fontsize=12)
ax.set_title('2023 Bahrain GP — Position by Lap', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.set_yticks(range(1, 21))
ax.grid(alpha=0.3)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()